In [18]:
import pandas as pd
import sklearn 
import os
import tensorflow as tf
tf.config.optimizer.set_jit(False)
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
dataset_path = "./PlantVillage/Pepper_Bell"

In [3]:
print(os.listdir(dataset_path))

['Pepper__bell___healthy', 'Pepper__bell___Bacterial_spot']


In [4]:
img_size = (224,224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

Found 2475 files belonging to 2 classes.
Using 1980 files for training.


I0000 00:00:1786118642.444012    1821 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3617 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:02:00.0, compute capability: 8.6


Found 2475 files belonging to 2 classes.
Using 495 files for validation.


In [5]:
class_names = train_ds.class_names

print(class_names)
print("Number of classes:", len(class_names))

['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy']
Number of classes: 2


In [6]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

val_ds = val_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

In [7]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)

val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [8]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [9]:
num_classes = len(class_names)

model = tf.keras.Sequential([
    data_augmentation,

    tf.keras.layers.Conv2D(32, 3, activation="relu", input_shape=(224,224,3)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(num_classes, activation="softmax")
])

/home/hp/ml/.venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [11]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50
)

Epoch 1/50


/home/hp/ml/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1786118647.052292    1935 cuda_dnn.cc:529] Loaded cuDNN version 92400


62/62 ━━━━━━━━━━━━━━━━━━━━ 17s 174ms/step - accuracy: 0.8778 - loss: 0.3034 - val_accuracy: 0.6626 - val_loss: 0.6474
Epoch 2/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 9s 147ms/step - accuracy: 0.9278 - loss: 0.1935 - val_accuracy: 0.6182 - val_loss: 0.7340
Epoch 3/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 9s 146ms/step - accuracy: 0.9530 - loss: 0.1452 - val_accuracy: 0.6182 - val_loss: 1.0699
Epoch 4/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 9s 147ms/step - accuracy: 0.9702 - loss: 0.0937 - val_accuracy: 0.6182 - val_loss: 1.0496
Epoch 5/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 9s 147ms/step - accuracy: 0.9737 - loss: 0.0796 - val_accuracy: 0.6081 - val_loss: 0.7313
Epoch 6/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 9s 147ms/step - accuracy: 0.9813 - loss: 0.0617 - val_accuracy: 0.6182 - val_loss: 1.5671
Epoch 7/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 9s 146ms/step - accuracy: 0.9879 - loss: 0.0429 - val_accuracy: 0.6283 - val_loss: 1.7098
Epoch 8/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 9s 146ms/step - accuracy: 0.9833 - loss: 0.0426 - val_accuracy: 0.9293 - val

In [12]:
"""
Layer	What it does
Conv2D	Applies learnable filters (kernels) to detect features like edges, corners, textures, spots, and patterns. It does not normalize values.
BatchNormalization	Normalizes the activations of the previous layer to stabilize training, allowing faster convergence and reducing internal covariate shift.
MaxPooling2D	Slides a small window (e.g., 2×2) over each feature map and keeps only the maximum value from each window. This reduces the spatial dimensions (height and width) while preserving important features.
GlobalAveragePooling2D	Takes each feature map (not each neuron) and computes its average. For example, a 7×7×128 tensor becomes a vector of 128 values—one average for each of the 128 feature maps.
Dense(128, ReLU)	A fully connected layer where every input connects to every output neuron through learnable weights and biases. It learns combinations of the extracted features.
Dropout(0.5)	Randomly sets about 50% of neurons to zero during training to reduce overfitting. It does not reduce the number of weights; it temporarily ignores some neurons while training.
Dense(num_classes, Softmax)	Produces one score for each class, and Softmax converts those scores into probabilities that sum to 1. The class with the highest probability is the prediction.
"""


'\nLayer\tWhat it does\nConv2D\tApplies learnable filters (kernels) to detect features like edges, corners, textures, spots, and patterns. It does not normalize values.\nBatchNormalization\tNormalizes the activations of the previous layer to stabilize training, allowing faster convergence and reducing internal covariate shift.\nMaxPooling2D\tSlides a small window (e.g., 2×2) over each feature map and keeps only the maximum value from each window. This reduces the spatial dimensions (height and width) while preserving important features.\nGlobalAveragePooling2D\tTakes each feature map (not each neuron) and computes its average. For example, a 7×7×128 tensor becomes a vector of 128 values—one average for each of the 128 feature maps.\nDense(128, ReLU)\tA fully connected layer where every input connects to every output neuron through learnable weights and biases. It learns combinations of the extracted features.\nDropout(0.5)\tRandomly sets about 50% of neurons to zero during training to 

In [13]:
val_loss, val_accuracy = model.evaluate(val_ds)
print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9939 - loss: 0.0146
Validation Loss: 0.01463726069778204
Validation Accuracy: 0.9939393997192383


In [14]:
from PIL import Image
import numpy as np

img = Image.open("./Leaf_1.jpg")
img = img.resize((224, 224))

img = np.array(img, dtype=np.float32) / 255.0
img = np.expand_dims(img, axis=0)

prediction = model.predict(img)

predicted_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction)

print("Prediction:", predicted_class)
print("Confidence:", confidence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
Prediction: Pepper__bell___Bacterial_spot
Confidence: 0.6180772


In [15]:
model.save("agrimodels/Pepper_Bell.h5")

In [16]:
os.listdir("agrimodels")

['Tomato.onnx',
 'Tomato.h5',
 'Potato.keras',
 'Potato.h5',
 'Tomato.keras',
 'Pepper_Bell.h5',
 'Potato.onnx']

In [17]:
os.listdir('PlantVillage')

['Tomato', 'Pepper_Bell', 'Potato']